# SciGeoGuard-X — Multi-Model Blind Validation v5.1

## Calibrated blind judges, judge-health gates, adaptive adjudication, and targeted human audit

This notebook fixes the failures observed in v5:

- Phi-3.5 `DynamicCache` failure;
- Judge A collapsing toward `REFUTED`;
- Judge C collapsing toward `UNPROVEN`;
- misleading consensus/FSCR numbers when a judge fails;
- ambiguity around `null` intent fields.

### Judge models

- **Judge A:** `Qwen/Qwen2.5-3B-Instruct`
- **Judge B:** `microsoft/Phi-4-mini-instruct`
- **Judge C:** `HuggingFaceTB/SmolLM3-3B`

All are evaluated locally in Colab, sequentially, with no paid inference API.

### New methodological safeguards

1. Balanced calibration set with equal `PROVEN`, `REFUTED`, and `UNPROVEN` examples.
2. Explicit schema semantics for `null`.
3. Judge eligibility gate based on:
   - calibration accuracy,
   - calibration per-class recall,
   - blind parse success,
   - severe verdict-collapse detection.
4. Broken judges are **excluded** from consensus.
5. Judge C is evaluated on all cases when fewer than two healthy A/B judges remain.
6. No consensus is reported unless at least two eligible judges support it.
7. Multi-model consensus remains separate from human expert ground truth.

## 0. Colab setup

Use **Runtime → Change runtime type → T4 GPU**.

The notebook checkpoints each judge independently and can resume from saved prediction CSVs if the runtime remains available.

In [ ]:
RUN_MODELS = True
MAX_CASES = 180
CHECKPOINT_EVERY = 5
MAX_NEW_TOKENS = 180
SEED = 20261002

# Judge health gates
MIN_CALIBRATION_ACCURACY = 0.75
MIN_CALIBRATION_CLASS_RECALL = 0.50
MIN_PARSE_SUCCESS = 0.95
MAX_SINGLE_LABEL_FRACTION = 0.95

# Human audit
HUMAN_AUDIT_TARGET = 50

MODEL_A = "Qwen/Qwen2.5-3B-Instruct"
MODEL_B = "microsoft/Phi-4-mini-instruct"
MODEL_C = "HuggingFaceTB/SmolLM3-3B"

from pathlib import Path
from datetime import datetime, timezone
import os, json, math, random, re, gc, shutil, gzip, base64, hashlib
import numpy as np
import pandas as pd

random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("/content") if Path("/content").exists() else Path("/mnt/data")
WORK = ROOT / "scigeoguard_v5_1"
WORK.mkdir(parents=True, exist_ok=True)

print("SciGeoGuard-X v5.1")
print("RUN_MODELS =", RUN_MODELS)
print("WORK =", WORK)

# Part I — Embedded blinded test set and frozen reference

The judge-facing table contains only:

- case ID,
- scientific intent,
- artifact semantics,
- workflow operations.

It does **not** contain generated labels, SciGeoGuard-X predictions, or error-family names.

In [ ]:
REVIEWER_PACK_GZIP_B64 = """H4sIAOx1v2oC/+1d25LbRpJ9n6/o6NcVNXXB1W/aUa9WMZLsmNbau+FwICASzcYKBLAAKKk9Mf++hQtJXAokwQYbAHkUlsfTKLLJqsysvJw8GTnfXOe7FUQLJ3o1t2PHchevvgfR1wcv+G65fuL4ySs7StwHe57E1v/Ggf8qTpyw+M8oe7kTWd+caOHOk90PnCgKImvu2XG8+2HsiHVu8rT7yTzwH9yF488di1pJYKnlR6uV+O2ll7uLv9BX9+/+m1v//uH9p7eW+IzBwlo4YfJoEUpf3f7z9nYZ2N7t7U83t7elp7e3r8QPIuf/1m7kLKz/W9t+Ij5Gus5fe17l4dp3k/KDhTN3YzfwrcRdOeUHK/uHFTlx4K2T9PFK8m6x9cWOHMsRG/iYPn6wvdipPJ97wXphrez4a+V5+t5rsSlRYoszqHzQxFmFQWR7Yrc8J7LTnXuUfpFvtrd2LLFiuf3Y/7p9dfu72CTfzr9K+m5RJH7FzCT5HpW35vbW8Zxvdvrt8oebrRGfL/9BPLe94o3EQSauvw7Wcf5oHsX5g7tf7t/9xJmm8M0pVLfMJJVvVXpc+VphFMydOHb9pSU+lOOVn8XrSIinY62ChVMc/tv7j/mvEwKXuOJjWgs7Wa+Kj/TuIyPEyBfYc7FfsZvIjtj+Zrue/cVzGk/Ex/nm+Onu52/5tyB0It+dr+Obt3fFr/5i+wsrCrzKC/1AfBBbaMIq3eh5XH7WcuK2J3Rxc6CVFwRfYif6Jh6tXF/+c/tH5Z2SJD+Wf/7rX69uKoLw3RaiMCt2UiYL2YLN3lelYd6jOFDy+gwCsXkgk4bf3t0biuV4nhvGgbvoUyoenxaRvfbc+U3xScYvFn8UJiIINwe5+uL6jrXZufw7hHZkr/KXiJen/5sd2pfsBAvZ2r2Dn764MKzNVxeXS/bS7OSDrY3JPk725y+s1fBzBYb/dMNPBzP8FIYfhn9Iwy+kwdRg7kdi7r+uWMXg87LBdwJr5aVf3V1kZskiilmz+vUlMP1tpv/u51lsr0LPiWecKaZM55P0bkg1rFhYVfyttk1Q7bdK0IvG2+vkMRARrZC4b85MPF2s58kU9T4OPam6pju6cRjyA8oXZQJiOf4if8gIU2ZEnxH6mZCfsn/+Lft3sdqJEytOhEqUl2uy5SULoOQW4D/vPrz9+b8+z/7jzcf3H/4n01VrIc7FzzR05cYrO5kLP5CwmkVoWQnD0GYYFq44I1/uBVSfnckhhCkYhQtw9AVOyhf4vBqwqc2ALXJWQeKIr+xnZ0eoLo3cqsugrm3qmu3WLP39MyaN4vLtdH6kSdyDN7hw++zoqaPKMijsSyusOJ2t3Fl2KO5tZ7GVv0Z0d09lgvHFnn8VR5+IwO6cnt1YZKPkn6ifqcQ/uQIrX5cLJpOLyHnwnLnknr8CuWDsKuWifvsXV1DlctvvAzTi/1WYXpzp9R1EcVsCgW4TCCx3IioXImW1dxXvaB/jjKwccQ5FULJ0hHjE7jx/XbBOwnVi7TzXesJBKzssnji0uTjiyJo/prewcFZ4zVmpL4Gj0uao/BZE3uJv6Va1OCrpVlrZXh42O3biLIWmbXNTVbujCMNzea7K3f2bm90mntus/J5a6HS7uPiriL+q+KuJv7r4a4i/acnWVF8JS07+6N/+5Cmnrsr+xfVcX+hAkXCwo6WTNDRHy17mfCsQB8GD5TspyuEhiFZF0vCnmyRaOyWzoB+OY4iKOKafOIaeM45ptw0UYcxYwphc+xDF7PVWKaIYRDFyudAQxYwmiuEvH8QY0ppJPHd84e4IufLsUAii/dVOQxpSr6FK18FxaXNc5o9uGMsskLtK9xe1U9ROS3q1KZSVVewhEnpfxB3iMLlZUmXzQD6CIh/x/HwEHSwfQZGPuOR8hBCh+Z9B3Hs6QuxFzSzEnni9JV7jzO04Ebe6LFGZLbrZLOrVNGz8T2NG1c9UkkW/CJvx/uPdP97NHhyhUJEUIJECYsTHamAkV1eHkCgJhCaFA9XNyac3wp68++XjTbbJE4xD0UVzrWDq+nURCoPizmXeYGE68huN/FELDTcBa9kM7sfOlu+Eoivu7d2vdx9+/uXj3afP1sM6/W3ly0SCkMsSq+e+FdKsxKXeCpnW41LodCm0YEQvNjzsYBJ2Gs2aGl2KNLLG3CgMvI1i112+1rUICttUeRs+o0Y9nXTOJEPDtEztbK7bPkND3jQaaz9eh0KaEmdR/h3LFL1PCGkg5ltXw3C0+wDxtgJZNRkbXRTKIPza5Ol8mWAYjlEUnYaGrhQY4J1FUFq7phWOrml0TW8D/c8I9NE1DbqMSfdPV4AA9Jj+KwO4xX5wi3yg/isOvw/AxekA1CgBcBHARYlc8GuVi3ECF5nSA3Ix9Gx/E9QexC1SbT/dC1VB9/J8uhddJWB7QaaqP8Qi1blR1mJ9P2ZRYcAsArM4hPJufopSVOWOnq+/bFoLWvLO/JRClNEsRMWhWC1ONzM9pRaEeglKug5moc0sVG5s3Oq41fff6rlyFfe6k67anaLKDSNXmNDxc23PhK+wA7pauenNA/46h7/+fH9d1cHOCM3uQbMJaddsSo2yZjMibSFcufnp7vZYXN71FqTmIqh5m5qv/a9+8N2fuX64TjpUk6HeG/UOHh7cuRD6iWn2739sNI1KNa2M3Vw5tm/Zy2XkLNsw3PuWQ/sOoj3PG0rjar2YEHujVo7VOPPj4+2Vs3BtvzrYgh3ZvKGheQPNGyNo3qAGmjf2Nm8wvj8LzlRkwcEkiCz4hWfBmXKoU5+xoTr1CTr1ca9XBOLKOvW1wQD8Gjr1r7dTn6ntYwxNNOSgIQf6PEBDDmYYoQHn+Q04f6+m9bSW6WXbvdmW1Jo8nKkyZnbL9m429vsS7gChHGr/d8Cj/acdLWQ6nz9Jy5Ff7LTLtt6CnY41K8bDeUIZYQbOnr0T+0LpC5XX9P0QFc4AUXk+RIUZBiAqgKj0SIKrmHr5HjUOaDGBFj9fi6nKoMZQ4165rA1WVuMDeFFGocY99HcxCjWGGvc0zlsRzl2Hcd7qgXHenBwgtDdQFkdzGMriF09hz2lXnkIDPIXgKYRzMAmewtda5Y9+Am0hZ9Lcefj4FGcHnDevfHN3/Mf1BHrbUliHNuvg+gvnh8w4fHr763tkzadmH2bkta5yQ9M1puiGZqqUycwFeW3onCmawUzdFP/Qgyj4OE5rYxWROsKKFB+KFiJdh0VwfnzDuIaGcTSMQ9FH0FZqcKq2t5Uq1CBlFT+MhqVAwwINOw40rPqZaIfQsOfPGmBSFfByl4V/5UewEhMOVuIJsBK3J4JBSgxS4gmRz6YtrSAlBikx5GLcpMSUvTQpMdeaaans5JaRm25svLKT+aOEk6W5CP5Ka1nKWUzR7NzefmBviqho7M5KaZPHevcEyaO8NvkicqBdixz4bjQyOWjmJMVpR9nY0m1BqmLeU8PSMPGZ+FiN1hznR/pu4gX5c8k+iBWeO3cTy/bcpb/KQiJJQVKXFiSr+MddiaLO7yFdhxuh7UaYP7qhtEDhrtL9RXUC1YlOUGTVVMq6fKChgKEtqAckssFUhhojtLg/LdYI4WU1NrtihBgwQsAIQa1LGCEpJIgNAwFSSDPXIt577dmpX+/YEgUuP4bS7q8HhYH3tNzYtheuB1Eo7PD38NmTpQrteCFTggsZFzL0uwLaNRjTFUPVNd2kOjUVOWhXV01OFI3rXKw2VWWgK5sdWR4hKI+gPILyyGTLIwzlEZRH9pRHFH4Y1UcpUH39oPrYOVF9msnRujl2VF+hf4D17Z0prwLWB1gf5OIqYH1LR4hHvBnicDhXpTQj17oQShoRmnIKd0XurgRhiXx3EMPz+jkxy+efEbv2b2c2m9mQgH0GZ6ez6oGpSgz0UZiqNIy23t2/udlt4kWPQTwnmdQpU5YUCUa/2OU/nSiQZJ9LT2ES9s1GRLf4ULmG7STtzTYPeHs/uJ5XaNQxOp99mKxsZIo/6U8yNXTjbSXYe8rBlDJl1o/yyyn88uv0yyP7u/X2E1zzsbnmRutUK/nUYky1wlQrsHScearVHC4Zxlo9f6zV11U1dWoervVy1HrB4IJaLxhcXoqpA6VelHplcmGg1DumUi9/+UqvKmkrelin9HblvLKIU3WZy3J24lF6ucSjWQAL3tFuvKNCHtRDvKMXZJY6cFDuNJoeQSFJEIAgAEEAggAEAQgCkGHlQkMAMpoARHtpBkmV7ScbUjjIhnoYe6oqIBsCyUFPY0/1GTM7jD3VD4w9VY9ojlM0xCsTaI7DYEN0x11KFxRBxIKIRSIXIL2/8ohF2R+xcA0Ry/MjFs00EbEgYnn2CEaqqrR9BCNnrMx8rKpShrbS97eKtgNJ72uKpMx+se3dbNT0ElRdqIHav6o/2n/akZS4KX+SdjV8sdOB2vVp6+Bpe2mFF/vCOTuryv/+x0YFtXakPANS/nSkPB8MKc/Pg5T/DKQ8kPJHmNlMGkytT1kAXP4ZcPm/V6Mp/QhmNAXJ336Sv5SA3xq5X+R+kftF7he5X+R+T4DLG8eniBp+C1JESBFdbIqIGMoLpYjMZsfK2o/XoTiSlKu5dCbLFL1CSF0P96xGGNHerhIncor1jUALiRIGrq6M/ToFUL8xXP6nMZylptp3NoXWFoqz4ogPcJzVCdY1SRdbiWhPvFzIbrgbdVe3CK1rYQ/a7MGWPbIlq/ASlIhILFwDM2JqM5xNi2mL2TBPYEbUjmiUYypyj8g9wmNAoxzI95F6HFYuOFKP18zUobE6v3vsCXfH2lBwSJyV7aKz83Sol8vT8f7j3T/egaijI1GHIhWIqlXavNGkLqYNoEobDFClgXp0auQslbtic4OVDd9+/Ez5FuD72w+ogfaD57cfGFynaD9Arru3hmneoWHaONAwrSktNWg3sbaFz/J4ad6of0lXwjK0WYaFG5dizqpJqD47E9c4bMEoMLNHI17JHoJoTT1qhgfDDI/rnOGBufDjG+ChaQdoigi87ud73Vw3DXjduGmPafqdO76z8bDEf1op0MCzQ+shEupb1HvJa9UwtLIa6/tHZDZIjjEiEyMyMSJzSiMylVOAIMYRQBAwJoOBDDc7utBejhuXAgoCKIhcLgAFGQ8UhLKXpiDTzAMUZBTZiOdnIxgnOrIRyEb0l41QdE0tqbEuaVoR0rn27MhKbYgkH1F+DCXeH2yEgfe03Ni6swUbCoKN0Sru2S9inR6CZDbYQF8MkkkByQQksy4Q/KIhmRfHcQdI5hQgmTrrQBtLwQkCTpCroY3NZeD8nCA6l/T/71LP7erXXISwqk357tlsi02SM4AAXnXZ8KoD9Zw+8rO7W1U52O5moN0NsdU42t00Kb7/gmIrc7DYykRsdcWxlXroFqAEtwBugek0Pd/efnpz/+bm3S8fb7JNniAg42LvBMyUmMSdoB26E5ThiDAo7gTcCWWBYJceGWCyEG6BIW6BA708HL08PfTygNsVvTzDkUGzE5p5dOOQd0gGyxgwAu8Q3mFVIIgJmrQp0aTBO5yEd2ju9w4VCu8Qnd5DeIfnt+aT9Afn6y8bJtwWZ1A7wRk0yAEzwGEGECQiSJysUeCnGAUqxeuu1l7i5p2g2dmVGRTr84Ja18JetKIHh+zuf43+fvT3T0oyTHT4H9+gX5pS39KgvzP+bH/vPNPRO99D7zxD7zxacHtl8hNnXVZjScNHHIrtEmeS/U5xmPZXOyP1M2saLV0HtW5T64qqQp2hzvvVOVeuQqGddFUpWjNNlitM6Ph5xJYJX5HTU7hSVnGlqeLizcRix/YtO7bs+XwtArHNTNd6Kqd1LVR9bw1PpuhhJL53mAmw+HLpzjareH997Fi72ZfWIQcmPYsLAdp+bm3f7r69XEbOcnvqxydwUvUrpCK7IazFurAH5VMUSuXZdVVMT6Oss2XLoB6YgQM27h58eKqaGi59mIHnXvqqkWfp5Zc+VfRKeK41L/2HdVolLid1JU59RukEdPfpmp+hNQDf6QDf0WecHtXwczGWoAOUY6fRurTaEj4+xdlh5LbgmxtsHHlad+TbluIWb9NlV9jZHzIl/vT21/fgRZna5T0jr7lmaITpikp1nXNqKjIVJq91SkzVICYVPrDBOT/IkBfH6USrilAd4eEXH4sWQl0HXxlH8Ok3WrTAp38inz4/J59+e4zOod1joV/J8Q1g099XU6UE1Xaw6cvlAmz642HT5y9Npm+YR2YcFGQckHEYvJ2cHtEwdOUZB1PCq7/243UoTjhxFuUjXqbTupuskHtWIyRpV+d461XK6SGFHIbiczydz8NQEZSMwZE4Y0swZXnN8ADke8dIWdgEKiGI3XUdiJcL2Q13Ocj69O3WtbAHbfZg20qDNrHp2INJtomkRsPZuOBtjSJq90YRU07svhWnHEC4wSsSWncj5AthMVo9CGceRAs50jBtshES5QJvALzB9oAKjU8DRcvxF+VQSZtRKmXeSoSlsGJhJZJG00V9edkU8E7+A9HgP/TnP6C/FP7D0HPF2Sn+g3JEHRRzxXuqg9KB5orDRmCu+KQqXioqoaiESuWCMVRCR1MJ1V66EmqqHesmDaQm6iYTrZsAzHXpdRP9tLKJ1mG0JcFoS4y2hF3odbCl2bV7gujonkD3BFSv0j1BuKIRg3JTY1RXiWbKuycMphoqNxVdo6ZumOow3ROm0anq2OA9QNURVUfcssNUHfUZ4X1WHSXY5Lf3H1NCk7efP0pc7t1D6HubvhfM9BhaBLU/hWW+PKOaEnJ8fNzAFCI+Rnx8mTe32Bdm0JeJkCmh3fxlAn8Z/jL85bH4y3qP/jIl7MhmPoJmPjTzjaGZT0Mz395mPkr4/hEuzMAIF0xywiSnaU32PGWYEyXKkbe7gdsdt/sIbndKcbsfuN3VQ9N6qTbUtN4UKItpvdDoikAYmNY7pWm995jWO4FpvZTIcYbZCId4vbK+u+KSXCc7+nhCVNm4B9laxH9tl0A+1WG2G+cw+qEPFHnd8Q99SPublztxaZn8QE+Z/ECJ3nl2p4HZnZjdidmdmN35MpKhSHP6mN152uxOrWL8JbziCydMHiVjO0tPYdcPQOFm5mBhn4mwbwxm/Lvw1qJZsZMyWcgWbPa+Kg3zCWeEbm9/e3dvKJbjeW4YB+6iT6l4fFpE9tpz5zfFJ5mkDf/i+o612bkjjHijFdpPX1wY1v0Zgdz+y5uTKTHbjb8C43+68aeDGX8K4z914z9p238GYYDNf4bN/3vV4lM5uL46JXw75rsBr5euw3XQdh3MH4UHJNN+d5XuL4Z9AtZb0quN2S+r2EMklLqA7ojDNPSKMtMDM3t1zOx9/sxezSAMM3uhxv2pMaNGRY3llNjVcylXX+q3cstKaPdI21Uvwiu/lGpLFohdU4R2xuzcJdVZusdrJYvO5RX1om15t22SrqnmItjxNju+9r/6wXd/5vrhOulgzeGcTQlXWelMppJ+hTh05km29eIrSEaPVZ5Dm9qbkBdTJO+9vf3A3hQX8thjnNImj9UfCpJHuT8EOehTDuLvbkS1kYlCM+ItDGeJvq/iBKW2pYFbzCTIamSknR/pu4kX5M93W+G7hbyJFZ47dxPL9tylv8qmCzQ71qjagT/VAD8M+GGuhR+GUu2lvDBtfwO5oqOBHA3kaCCfVgO5ckoDOdVbMUScAUMEACkwRMAQAUN0UbhRauwHHigAHvQAPKAa1wA8QDWqL1pA2istIDW7TrZSMdnqMiZbqTAKFz7ZiqrkhMlWlJGuneSUoJMcneToJEcn+ctIBv9MODrJ++okp6xi/WnTIxSWa+3ZaTXQsSUEQ+XHMPD7B7KHgfe0DPwzD2RXMMx0tDp79mnFlLHWZD5FMh8NwUjmI5k/oCVHFr/PLD7jnXqBCXqB0QsMH240TYSqWVFmpem5Rc4qSPIQL0NDKqbUhasugwbvD8PS39+CzOopDNNMzi8NljVZDRaHsxU7yw6F9jqLHQyqnm8bMhM7vmxbyu2OPOzQedhRygVjyMKWGN7LV9uhSbm1YOCoNC7dRhNMksZlSu1dT0k0LR0hHrE7PzrVpLZzz5lINSHVNNQYbqSagBtFxukM3HNM6zZUWsNQaQyVRrw6CvSoNqO8T/Qok88cCR+f4uxQM9G2vrmB1zZdum0p7EGbPSg1cVetwae3v75H++7UbMKMvDZkNoG+5geBBHGc3ucVqTki2it+Ly2ktkF7xNpnSSgUIR1aARHSIaRDSHdJIZ3ZkbqUgLoU1KWgLgV16amyYGqQhP4JSzlpQvdL5DFWOn40CrfBeAML1roWhrzNkG8ZkUDzMx3VnST3j5Ck+Z9BvLfhUzuF+ofTLgBSogBACgApUndjAZAqRKkoMzsMICU6AKT9AEjpMABSCg0GgHQ6QEECACkApHK5AIB0RABSbQD8KOfd0EQq0ERAE8F5GQmaSAIPegaaiCv76cgb3AWgIwcdOejIx52SPImNnKtdWSkVsFJeBislihkXz0pZuIBdaSm51qlOoaFOgToFLMFY6hTc1CrKrB9Rp2CoU0yA6ALAhAnUKYrbFIUKFCpQqOgsF9q1ysUFM110o1TlRjMjUZzSn04USMZVl57CR2nPOMhB8OnPxadp4p5BotmjSUmbAeOdEz2oZXlwPa/QqGM0Ofsw2/+XqaAbbxuPvad8SpU0t2jurzJwVBlQZRhIqe/u39zsNhEw6BNrDuopNQdF0jwRCSG0Vo7tW3Zs2fP5erXeMhnUKw6ta2Ev2uzF+493/3gnsxNhJL53mF1u4sulO9t0BP762NEVaDcZQr7JfqtBXqvIVJzbBdjuvr1cRs5ye+rHG4e0aWm5E5fcwliLdWQ3jlJolmfX9TE9krLiVuwDPZyxpMhYTgFZvc8SIGMJZDWoeZGwnLZc6EhYXhqyulu+UmH7sxwNIDWyHMhyIMsxKSDVSdBKhTfTHNnmLyN3Ueb4qdczmotgItq7LBZT9Edubz+wN0XoPPYoprTJY3VKg+RRjqaFHPQpB74bjUwOmiA6cdpRRqqz5W6t3AWpYWncB5n4WA0ON+dH+m7iBflzyT6IFZ47dxPL9tylv8pyJZKLQGml+eQcNJ+g+QTNJ2g+QfM5ZZrPVS0n0D6rR1Fh8TGrB2OhYfFh8S/J4mudCEEbDTcgBH0GISgfLDnMUcW+ghxx3oK3Jz9snpQf1o/hgl/7X/3guy9JE8sXwl6ACR5M8GCChyy8FBe8YnSkUCEmKFQug0IF7t+lU6hoJxGoKGY3rkUGrkVwLcISjIJrUe+Za1ElXRnWGNyDy3APVBiFi2dYO8k/UGnTJmyhK+nOSnyCynPoPQCBAAQCEAhA4BQBgSprhYcwAnjI6fAQPhg8hAMQOHV4yNdJJ/h/e3dvKJbjeW4YB+4COJGR4EQa/YIq70SuTEGuDHJlJAHGQq5MKqqsHA7jCcJ4hPEI4xHGj1sOnMXGi0Ao3ymUVw9zVTFwVfXEVcWHmQIMmAfY9SdFSkRAVgWyKqlcYAzwiMiqKBtgDrCq1fmqYs9dONZDEDlzO04swhUJY1W26GazqFdvZSOgRgqloRIBvRwK3tmDI1Q2csDEv8+XqQgEMZsCUbdLn97cv7l598vHmxLP8aQuqotltEB/c8ebKRQGxZ3LcpWF6cjb6Mgftatjc6OVzeD+KkXlUtAPXgpkqEshpV7FpYBLYScQilQgXnqQ9BmvAW2wa0DDNXDN14Cxn8uW6uCyfT6XLR2MrmC6pPvnN+eXy16rncJOoHZrYaMqWtjQwoYqxyha2LQZU/psYdNIO16ZAq8MAlP4+aCz68xHYmoAKY8GpFwz+LTV4FMFBh8GHw0qaFCBxZ+wxf97zeAzGZJd2KRtAL/rSGk2pkvW4SpouwoqLSe1QD+9I1JNQ1sKov2KchWG30lXlU7RYHquMKHj58m9TPgK1KqmVJL7mrzxLJumHa9X1ndX7Oo62Q3xbap661qoe5u65wO2Z7tRyZi/Db1//vzteL06w+RtrTKpyAmslZfuhLsoDAKvM1fWl8AOtNmBu59nxa0+07mu4vqHGeivK1VVK1qsSi96ofKJm+OVsy0ujaBsJHVa10LB2xR80B6U1yPsNkjB3OhCGboLZZSSwdCH0qmNpJT/kbSRMKVi/bWufKMEfKOXwTdKCby6CyccVQuD3pVxVNMP96krFH3q/fSp03P2qU+i6fTqG9VzPCX61BEhoE+9s1zwa5WLcfap8wHa1DWjFYJEdECQMEIZECRAkDBD+ZJmKGvm/u5DRUH3IboPh4lO7+7f3Ow2EVOST5mSzE7pQ9TJfkhCo3QJSMIJkASVMwWQBCSvn4tI1AxFa0ckUkMzK7pNpUCF0gakuIM8OU3ryenUDc9+s+3dbPT0EnRd6IHav64/2n/akZRSO3+SVqe+2GkJol6gEjsjNsRPt0YYgPjaaGcG0HixL1xVzqrzv/+x1cE9s4g05FnQ6oU8C/IsaPW6pAlEOt+fZ2nMHEKe5YQ8Cxssz8KQZ7nkPEuKEXI2bJt9Uj7pymGMEFeBEeoHI8SGmWXBEHNhlgVmWQAjhFkWwAhNHCOkq81WB3H/rj07nfjl2CKUqTeslh/DTdnvpoSB97TcZKpeGsqMmVsjMC2nKHEoQtxNE8MRKqztL+8SlHd7KO8aqspR3oVC90Mvq8wo70Avqx6il9V1aRlYiGpajbSEcbGz7fSCOJZkJ6XrYBbaiaZrEYO8W/Hq2pOvKB2x0RjPfkrTCVl0+bcPhVJW7MUiCsKGIh7jEWxlchF89+NE+BEraV+ibnTmquDgqgBXBTrRwFXxYpLBkWfqi6uCV4y/fMZIleloSzzaQABK18Hotxn9+aMbSmlH3VW6vwgBEQJ2Ih3jVCsrs0EOjZAkGkZIYoTkOEZIatIExsXOFeaDgVE5wKjXO1Dy/wEu5U1ouX4DAA=="""
PILOT_GZIP_B64 = """H4sIAOx1v2oC/61a247bNhB977e4AC+SZT+2sNMG2EuQeIP2iWAkWhZWJlVK2jT5+lLuer2+SBqO+LQGFsPLXM45M1QqayWKbLa15qfSorIqK9KmMHqWK62sbFQm1L+VSt2P2V426U7V4vSvUn5T5UyLbaGzQuf1L1/++IuL3+8+PqzEtjQmE5mqmp0glM4+rz88bdart78b26oZ7bXg0W0L9t5CGbEvxYssi0x2pxYkWg5u9Of6bvX4tPn1w2/3H+/+Fq0uGpEVe6Xrznpf1Ic7CkLY7NPnx6/rh+OfwyLk+rRW7U2jRN2toHN30QRw0VLqLDUvyop0J3WunBkH++dyRxL73LhOXfBEt3cpK1Eq+Sy7/ckSd2wKNKtdhJTYGqtSWTduPz4Y3dX66/ru8dP9+mEjtm3TWpek2lnvX6M8GJ73xqlL09zYIpWlW6FRtjLlcQ0OXKPVdVtVxnYJb1VtyvawQC6LbhUCTJTXtI7wgaYLgOl1RdAYFdqIgSNUV24r5+C6Kl05nXJq3DU3TjsUlYtcduV68Exl3cm11Gm3KYXbv0+OvZJayDy3Kj/l2JDfRlN0jnI7iwGwd6uiGAMYniPsctDiCipTZRuX882PN8d3sOEfYs5QecwJyozR0SPewLUFOPmH0WHhg83V7kd9SEZ7OMVLccKqJRCr+kpxDnLCVU5RcE5dsRIHV883t7PIbZGd828AUotxGcMCBQ0OIM60LaUV0irpe/n+A1ACzJqbESB4oqLgizvoTtX/YFKqF1WOJk4vbsKdrY2LtBQ/lTUjJDVyUm9N68rw6eF1n7cfUK9yOo2PErzCxCHvBKkTzVHin8/9xP8tRuN+eo4wNETSaDL/uiVCaNgogJqOkA2TEzwoaopRGeKIHSwP+5tDjscMhhBMEUFB4oSKZ7iK94CoC8ZLUEnQV+/co94pNPdL0zp+lPXzmTGu/WWLwYP3G1KCclQE7SrOM4gn2PaegIjuRtdJUQcdoZrLvrEtm6LLdWNF4+r7fWlHiPpkCTjt+/Q5VN93Y7fXXrUWMk1bd5fTPAM1HoAKoJuiYgmH0wGdypHcEcXTJFEY9oSD3gB/Ugb3ZKP27jBvWXTsezrdPZ3H53jRRsI0zZRPV3Fj+ANv4pLpcfHohVdf7rvKXm3uR1oTkDoMkFMj2n+0xAhi+sIW04bDCxRHUi8xcQDiut2L70WzM60Tiq097h97jDAHqGjh21728VD/CJBFAcYsFDfPG3kpudjbbdocKtSZO69jJ0U3R8ZjYHFO4CptujqxplSj4A/CqgWiRCLvUCOnrhFc2wyDepiioATbYcR+fT1lPs15X2XgZ2jR0u+8zOvdsxft5yH0nMfo8PgqRsNgAQkixUiI2XOEH7clQUIZY949WCAJF4JXJkhRAr/H+TSYoiYtHD58HmrgsCM0ylBNM4mnPdFQP4TinpPVKA7QzXgo4AtUafWzNt+1x4RouEtcBtDiLBAXM/Br4rnYYZ6cRAJwKNj9l4fFcy9jyLEXj7CGBFXBNPFxcO/YwFMaMe93r5GXDsiAzEvlDzVnyI8PloEGjCPMCK/hCdLS/90ywbQomFHq4IMWaObhyUpsjuNOiuRqHoNL4aKBwXxqQzzc+U/rDDtXqkYeJFFp6trvg6qhvPf4rquvm0J/bNgrJP8DBIZ2cf8qAAA="""
FROZEN_V3_RECORD = {
  "frozen_at_utc": "2026-10-02T08:19:38.747739+00:00",
  "verifier_version": "SciGeoGuard-X-v3",
  "v3_notebook": "/mnt/data/SciGeoGuard_X_Publication_Experiments_v3.ipynb",
  "v3_results_zip": "/mnt/data/SciGeoGuard_X_v3_All_Results.zip",
  "v3_notebook_sha256": "afdb1ad9c2cb8742b3d26638067e80fb8619afd6aa019a58769e5acbdc2c15d2",
  "v3_results_zip_sha256": "8cb3f200453e8018f5afd0ce3c95130f6efdee4733f819727b5b16fc159ef992",
  "freeze_status": "VERIFIER_NOTEBOOK_HASHED"
}

def decode_csv_b64(payload):
    from io import BytesIO
    raw = gzip.decompress(base64.b64decode(payload.encode()))
    return pd.read_csv(BytesIO(raw))

blind_cases = decode_csv_b64(REVIEWER_PACK_GZIP_B64).head(MAX_CASES).copy()
frozen_reference = decode_csv_b64(PILOT_GZIP_B64).copy()

blind_cases = blind_cases[[
    "review_order",
    "case_id",
    "workflow_intent",
    "artifacts_json",
    "steps_json"
]].copy()

assert blind_cases.case_id.nunique() == len(blind_cases)
assert "generated_expected" not in blind_cases.columns
assert "frozen_prediction" not in blind_cases.columns
assert "family" not in blind_cases.columns

print("Blind cases:", len(blind_cases))
print("Frozen verifier status:", FROZEN_V3_RECORD["freeze_status"])
display(blind_cases.head(3))

# Part II — Install model runtime

The three models are loaded one at a time.

v5.1 uses current `transformers` rather than pinning the old Phi-3.5 stack. Phi-4-mini is used in place of Phi-3.5.

In [ ]:
if RUN_MODELS:
    import sys, subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.55,<5",
        "accelerate>=1.0",
        "bitsandbytes>=0.45",
        "huggingface_hub>=0.26",
        "sentencepiece"
    ])

    import torch

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA GPU is required for the full v5.1 run. "
            "Choose a T4 GPU in Colab Runtime settings."
        )

    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU RAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("RUN_MODELS=False: dependency/model execution skipped.")

# Part III — Record exact model revisions

In [ ]:
MODEL_REGISTRY = {
    "judge_A": {"model_id": MODEL_A},
    "judge_B": {"model_id": MODEL_B},
    "judge_C": {"model_id": MODEL_C},
}

if RUN_MODELS:
    from huggingface_hub import model_info

    for name, rec in MODEL_REGISTRY.items():
        try:
            info = model_info(rec["model_id"])
            rec["revision_sha"] = info.sha
            rec["pipeline_tag"] = getattr(info, "pipeline_tag", None)
        except Exception as e:
            rec["revision_sha"] = None
            rec["revision_lookup_error"] = repr(e)

(WORK/"model_registry.json").write_text(
    json.dumps(MODEL_REGISTRY, indent=2),
    encoding="utf-8"
)

MODEL_REGISTRY

# Part IV — Scientific review rubric

The most important correction from v5 is the treatment of `null`.

### Schema rule

A `null` optional intent field means:

> **No additional constraint of that type was declared by the intent.**

It does **not** automatically mean missing evidence.

Examples:

- `required_unit = null` → do not demand an output unit merely because this field is null.
- `max_uncertainty = null` → no maximum uncertainty threshold was declared.
- `temporal_tolerance_h = null` → do not call the case UNPROVEN solely for this null field.
- Missing metadata inside an **artifact** can still make a specific scientific operation UNPROVEN when that operation genuinely requires the metadata.

In [ ]:
SYSTEM_PROMPT = """
You are an independent scientific reviewer for Earth-observation and geospatial workflows.

Your task is to classify the supplied workflow as PROVEN, REFUTED, or UNPROVEN using only the supplied
scientific goal, artifact metadata, and operations.

VERDICT DEFINITIONS

PROVEN:
No supplied scientific constraint is violated, and the supplied evidence is sufficient for every operation
that is actually required by the declared goal/workflow.

REFUTED:
At least one supplied operation or assumption is scientifically incompatible with the declared workflow.
Examples include physically incompatible units, future information leakage, invalid categorical
interpolation, incompatible vertical datums, scientifically invalid area calculation, or an explicitly
wrong spectral role.

UNPROVEN:
No contradiction is established, but a piece of evidence that is genuinely necessary for an operation
cannot be established from the supplied metadata.

CRITICAL NULL SEMANTICS

Optional intent fields with null values do NOT mean evidence is missing.
They mean that the intent does not impose an additional constraint of that type.

For example:
- required_quantity = null means there is no extra required-quantity constraint.
- required_unit = null means there is no extra required-unit constraint.
- max_resolution_m = null means there is no extra maximum-resolution constraint.
- max_uncertainty = null means there is no extra uncertainty-threshold constraint.
- temporal_tolerance_h = null means there is no extra intent-level tolerance constraint.

Do not assign UNPROVEN merely because an optional intent field is null.

Only call missing metadata UNPROVEN when a workflow operation itself scientifically requires that metadata.

SCIENTIFIC PRINCIPLES

- Execution success is not scientific validity.
- Categorical classes must preserve categorical identity.
- Rates and accumulations are different physical quantities.
- CRS and geometry determine whether area calculations are meaningful.
- Acquisition/availability time can create temporal leakage.
- Vertical datum compatibility matters when vertical quantities are combined.
- DSM and DTM have different semantics.
- Missing provenance may prevent certification.
- Train/test dependence can invalidate evaluation.
- Missing uncertainty matters only when the workflow/intent actually requires an uncertainty guarantee.

OUTPUT

Return one JSON object only:
{
  "verdict": "PROVEN" | "REFUTED" | "UNPROVEN",
  "error_class": "short category or NONE",
  "severity": "LOW" | "MEDIUM" | "HIGH" | "CRITICAL",
  "confidence": 1 | 2 | 3 | 4 | 5,
  "reason": "one concise scientific explanation"
}
""".strip()

def make_case_prompt(row):
    return f"""
Scientific intent:
{row['workflow_intent']}

Artifacts:
{row['artifacts_json']}

Workflow operations:
{row['steps_json']}

Apply only the constraints that are actually stated or scientifically required by these operations.
Return the required JSON object.
""".strip()

print(SYSTEM_PROMPT[:1800])

# Part V — Balanced calibration cases

These are small synthetic examples created **outside the 180-case test set**.

There are equal numbers of:

- PROVEN,
- REFUTED,
- UNPROVEN.

They teach the three-state semantics without exposing any blind-test label.

In [ ]:
CALIBRATION_CASES = [
    # -------------------- PROVEN --------------------
    {
        "cal_id":"CAL_P_01",
        "workflow_intent":'{"goal":"resample nominal land cover","required_quantity":null,"required_unit":null}',
        "artifacts_json":'[{"name":"landcover","quantity":"land_cover","scale":"categorical","resolution_m":10,"provenance":"official"}]',
        "steps_json":'[{"op":"resample","params":{"artifact":0,"method":"nearest","target_resolution_m":30,"evidence_of_new_information":true}}]',
        "truth":"PROVEN"
    },
    {
        "cal_id":"CAL_P_02",
        "workflow_intent":'{"goal":"compute daily rainfall accumulation","required_quantity":null,"required_unit":null}',
        "artifacts_json":'[{"name":"rain","quantity":"precipitation_rate","unit":"mm/h","temporal_resolution_h":0.5,"provenance":"official"}]',
        "steps_json":'[{"op":"temporal_aggregate","params":{"artifact":0,"method":"integrate","sample_duration_h":0.5,"declared_quantity":"accumulation"}}]',
        "truth":"PROVEN"
    },
    {
        "cal_id":"CAL_P_03",
        "workflow_intent":'{"goal":"combine elevations","max_uncertainty":null}',
        "artifacts_json":'[{"name":"a","quantity":"elevation","unit":"m","vertical_datum":"EGM2008","provenance":"A"},{"name":"b","quantity":"water_level","unit":"m","vertical_datum":"EGM2008","provenance":"B"}]',
        "steps_json":'[{"op":"combine_vertical","params":{"a":0,"b":1}}]',
        "truth":"PROVEN"
    },

    # -------------------- REFUTED --------------------
    {
        "cal_id":"CAL_R_01",
        "workflow_intent":'{"goal":"resample nominal land cover"}',
        "artifacts_json":'[{"name":"landcover","quantity":"land_cover","scale":"categorical","resolution_m":10,"provenance":"official"}]',
        "steps_json":'[{"op":"resample","params":{"artifact":0,"method":"bilinear","target_resolution_m":30,"evidence_of_new_information":true}}]',
        "truth":"REFUTED"
    },
    {
        "cal_id":"CAL_R_02",
        "workflow_intent":'{"goal":"daily rainfall accumulation"}',
        "artifacts_json":'[{"name":"rain","quantity":"precipitation_rate","unit":"mm/h","temporal_resolution_h":0.5,"provenance":"official"}]',
        "steps_json":'[{"op":"temporal_aggregate","params":{"artifact":0,"method":"mean","sample_duration_h":0.5,"declared_quantity":"accumulation"}}]',
        "truth":"REFUTED"
    },
    {
        "cal_id":"CAL_R_03",
        "workflow_intent":'{"goal":"calculate flood area"}',
        "artifacts_json":'[{"name":"mask","quantity":"flood_extent","scale":"binary","crs":"EPSG:4326","provenance":"official"}]',
        "steps_json":'[{"op":"area","params":{"artifact":0,"method":"planar","output_unit":"km2"}}]',
        "truth":"REFUTED"
    },

    # -------------------- UNPROVEN --------------------
    {
        "cal_id":"CAL_U_01",
        "workflow_intent":'{"goal":"combine elevations"}',
        "artifacts_json":'[{"name":"a","quantity":"elevation","unit":"m","vertical_datum":"EGM2008","provenance":"A"},{"name":"b","quantity":"water_level","unit":"m","vertical_datum":null,"provenance":"B"}]',
        "steps_json":'[{"op":"combine_vertical","params":{"a":0,"b":1}}]',
        "truth":"UNPROVEN"
    },
    {
        "cal_id":"CAL_U_02",
        "workflow_intent":'{"goal":"certify source provenance"}',
        "artifacts_json":'[{"name":"input","quantity":"elevation","unit":"m","provenance":null}]',
        "steps_json":'[]',
        "truth":"UNPROVEN"
    },
    {
        "cal_id":"CAL_U_03",
        "workflow_intent":'{"goal":"make forecast","decision_time":"2024-08-01T12:00:00+00:00"}',
        "artifacts_json":'[{"name":"feature","quantity":"rainfall","unit":"mm","available_time":null,"acquisition_time":null,"provenance":"official"}]',
        "steps_json":'[{"op":"predict","params":{"features":[0]}}]',
        "truth":"UNPROVEN"
    }
]

calibration_df = pd.DataFrame(CALIBRATION_CASES)
display(calibration_df[["cal_id","truth"]])

# Part VI — Robust parser

In [ ]:
VALID_VERDICTS = {"PROVEN","REFUTED","UNPROVEN"}
VALID_SEVERITIES = {"LOW","MEDIUM","HIGH","CRITICAL"}

def extract_json_object(text):
    if not isinstance(text,str):
        return None

    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.I)
    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        x = json.loads(cleaned)
        if isinstance(x,dict):
            return x
    except Exception:
        pass

    decoder = json.JSONDecoder()

    for m in re.finditer(r"\{", cleaned):
        try:
            x, _ = decoder.raw_decode(cleaned[m.start():])
            if isinstance(x,dict):
                return x
        except Exception:
            continue

    return None

def parse_judge_output(text):
    obj = extract_json_object(text)

    result = {
        "verdict":"UNPARSED",
        "error_class":"",
        "severity":"",
        "confidence":np.nan,
        "reason":"",
        "raw_output":text
    }

    if obj:
        v = str(obj.get("verdict","")).strip().upper()
        if v in VALID_VERDICTS:
            result["verdict"] = v

        result["error_class"] = str(obj.get("error_class","")).strip()

        sev = str(obj.get("severity","")).strip().upper()
        if sev in VALID_SEVERITIES:
            result["severity"] = sev

        try:
            conf = int(obj.get("confidence"))
            if 1 <= conf <= 5:
                result["confidence"] = conf
        except Exception:
            pass

        result["reason"] = str(obj.get("reason","")).strip()

    if result["verdict"] == "UNPARSED":
        m = re.search(
            r'(?:verdict|decision)\s*[:=]\s*["\\\']?(PROVEN|REFUTED|UNPROVEN)',
            text or "",
            flags=re.I
        )
        if m:
            result["verdict"] = m.group(1).upper()

    return result

# parser smoke test
assert parse_judge_output(
    '{"verdict":"PROVEN","error_class":"NONE","severity":"LOW","confidence":5,"reason":"constraints satisfied"}'
)["verdict"] == "PROVEN"

print("Parser smoke test passed.")

# Part VII — Model loader

v5.1 uses model-native chat templates and sequential GPU loading.

If FP16 fails due to memory, the loader retries with 4-bit NF4 quantization.

In [ ]:
def load_model(model_id):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    print("Loading", model_id)

    tok = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=True
    )

    common = dict(
        pretrained_model_name_or_path=model_id,
        trust_remote_code=True,
        device_map="auto",
        low_cpu_mem_usage=True
    )

    try:
        model = AutoModelForCausalLM.from_pretrained(
            **common,
            torch_dtype=torch.float16
        )
        mode = "fp16"

    except RuntimeError as e:
        if "out of memory" not in str(e).lower():
            raise

        print("FP16 OOM; retrying 4-bit NF4.")
        gc.collect()
        torch.cuda.empty_cache()

        qcfg = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )

        model = AutoModelForCausalLM.from_pretrained(
            **common,
            quantization_config=qcfg
        )
        mode = "4bit_nf4"

    model.eval()
    return tok, model, mode

def unload_model(tok, model):
    import torch
    del tok
    del model
    gc.collect()
    torch.cuda.empty_cache()

# Part VIII — Deterministic inference

In [ ]:
def render_messages(tokenizer, row, calibration=False):
    user_prompt = make_case_prompt(row)

    # Keep the blind test prompt and calibration prompt structurally identical.
    messages = [
        {"role":"system","content":SYSTEM_PROMPT},
        {"role":"user","content":user_prompt}
    ]

    try:
        rendered = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    except Exception:
        rendered = (
            "SYSTEM:\n" + SYSTEM_PROMPT +
            "\n\nUSER:\n" + user_prompt +
            "\n\nASSISTANT:\n"
        )

    return rendered

def generate_one(tokenizer, model, row):
    import torch

    rendered = render_messages(tokenizer,row)

    inputs = tokenizer(
        rendered,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )

    device = next(model.parameters()).device
    inputs = {k:v.to(device) for k,v in inputs.items()}

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

# Part IX — Calibration gate

A judge is eligible only if it demonstrates that it understands all three verdict states on the balanced calibration set.

In [ ]:
def calibration_metrics(df):
    if df.empty:
        return {
            "accuracy":0.0,
            "macro_recall":0.0,
            "min_class_recall":0.0,
            "parse_success":0.0
        }

    parsed = df.verdict.isin(VALID_VERDICTS)
    accuracy = float(
        (df.loc[parsed,"verdict"] == df.loc[parsed,"truth"]).mean()
    ) if parsed.any() else 0.0

    recalls = []
    for label in ["PROVEN","REFUTED","UNPROVEN"]:
        g = df[df.truth==label]
        recalls.append(
            float((g.verdict==label).mean()) if len(g) else np.nan
        )

    return {
        "accuracy":accuracy,
        "macro_recall":float(np.nanmean(recalls)),
        "min_class_recall":float(np.nanmin(recalls)),
        "parse_success":float(parsed.mean()),
        "recall_PROVEN":recalls[0],
        "recall_REFUTED":recalls[1],
        "recall_UNPROVEN":recalls[2]
    }

def run_calibration(judge_name, model_id):
    path = WORK/f"{judge_name}_calibration.csv"

    tok, model, load_mode = load_model(model_id)

    rows = []
    try:
        for _,r in calibration_df.iterrows():
            try:
                raw = generate_one(tok,model,r)
                parsed = parse_judge_output(raw)
            except Exception as e:
                parsed = {
                    "verdict":"UNPARSED",
                    "error_class":"",
                    "severity":"",
                    "confidence":np.nan,
                    "reason":"",
                    "raw_output":"",
                    "inference_error":repr(e)
                }

            rows.append({
                "cal_id":r.cal_id,
                "truth":r.truth,
                "judge":judge_name,
                "model_id":model_id,
                "load_mode":load_mode,
                **parsed
            })
    finally:
        unload_model(tok,model)

    out = pd.DataFrame(rows)
    out.to_csv(path,index=False)

    metrics = calibration_metrics(out)

    eligible = (
        metrics["accuracy"] >= MIN_CALIBRATION_ACCURACY and
        metrics["min_class_recall"] >= MIN_CALIBRATION_CLASS_RECALL and
        metrics["parse_success"] >= 0.95
    )

    metrics["eligible_after_calibration"] = bool(eligible)

    print(judge_name, json.dumps(metrics,indent=2))

    return out, metrics

# Part X — Run calibration for A, B, C

All three judges are calibrated **before** blind-test predictions are used.

In [ ]:
CAL_RESULTS = {}
CAL_METRICS = {}

if RUN_MODELS:
    for judge_name, model_id in [
        ("judge_A",MODEL_A),
        ("judge_B",MODEL_B),
        ("judge_C",MODEL_C)
    ]:
        cdf, cm = run_calibration(judge_name,model_id)
        CAL_RESULTS[judge_name] = cdf
        CAL_METRICS[judge_name] = cm
else:
    print("RUN_MODELS=False: calibration skipped.")

calibration_summary = pd.DataFrame([
    {"judge":j,**m}
    for j,m in CAL_METRICS.items()
])

display(calibration_summary)

# Part XI — Resumable blind runner

Blind predictions are generated independently.

Judge eligibility is assessed again after the blind run so a model with severe parse failure or verdict collapse cannot enter consensus.

In [ ]:
def run_blind_judge(judge_name, model_id, order_seed):
    out_path = WORK/f"{judge_name}_blind_predictions.csv"

    if out_path.exists():
        old = pd.read_csv(out_path)
        rows = old.to_dict("records")
        done = set(old.case_id.astype(str))
        print(judge_name, "resuming from", len(done))
    else:
        rows = []
        done = set()

    order = blind_cases.sample(
        frac=1,
        random_state=order_seed
    ).reset_index(drop=True)

    remaining = order[
        ~order.case_id.astype(str).isin(done)
    ]

    if len(remaining)==0:
        return pd.DataFrame(rows)

    tok, model, load_mode = load_model(model_id)

    try:
        for i,(_,r) in enumerate(remaining.iterrows(),start=1):
            try:
                raw = generate_one(tok,model,r)
                parsed = parse_judge_output(raw)

                rec = {
                    "case_id":r.case_id,
                    "judge":judge_name,
                    "model_id":model_id,
                    "load_mode":load_mode,
                    **parsed
                }

            except Exception as e:
                rec = {
                    "case_id":r.case_id,
                    "judge":judge_name,
                    "model_id":model_id,
                    "load_mode":load_mode,
                    "verdict":"UNPARSED",
                    "error_class":"",
                    "severity":"",
                    "confidence":np.nan,
                    "reason":"",
                    "raw_output":"",
                    "inference_error":repr(e)
                }

            rows.append(rec)

            if i % CHECKPOINT_EVERY == 0 or i == len(remaining):
                pd.DataFrame(rows).drop_duplicates(
                    "case_id",
                    keep="last"
                ).to_csv(out_path,index=False)

                print(
                    judge_name,
                    f"{len(rows)}/{len(order)} saved"
                )

    finally:
        unload_model(tok,model)

    return pd.read_csv(out_path)

def blind_health(df, calibration_metric):
    if df.empty:
        return {
            "parse_success":0.0,
            "max_label_fraction":1.0,
            "eligible":False
        }

    parsed = df[df.verdict.isin(VALID_VERDICTS)].copy()
    parse_success = float(len(parsed)/len(df))

    if len(parsed):
        dist = parsed.verdict.value_counts(normalize=True)
        max_label_fraction = float(dist.max())
    else:
        max_label_fraction = 1.0

    calibration_ok = bool(
        calibration_metric.get("eligible_after_calibration",False)
    )

    eligible = (
        calibration_ok and
        parse_success >= MIN_PARSE_SUCCESS and
        max_label_fraction <= MAX_SINGLE_LABEL_FRACTION
    )

    return {
        "parse_success":parse_success,
        "max_label_fraction":max_label_fraction,
        "calibration_ok":calibration_ok,
        "eligible":bool(eligible)
    }

# Part XII — Run Judge A and Judge B

If a judge fails calibration, it is still allowed to run for diagnostic purposes, but its outputs are not automatically trusted for consensus.

In [ ]:
if RUN_MODELS:
    judge_A = run_blind_judge("judge_A",MODEL_A,101)
    judge_B = run_blind_judge("judge_B",MODEL_B,202)
else:
    judge_A = pd.DataFrame()
    judge_B = pd.DataFrame()

HEALTH = {}

if not judge_A.empty:
    HEALTH["judge_A"] = blind_health(
        judge_A,
        CAL_METRICS.get("judge_A",{})
    )

if not judge_B.empty:
    HEALTH["judge_B"] = blind_health(
        judge_B,
        CAL_METRICS.get("judge_B",{})
    )

print(json.dumps(HEALTH,indent=2))

# Part XIII — Decide where Judge C is required

- If both A and B are healthy → C judges only A/B disagreements or parse failures.
- If either A or B is unhealthy → C judges **all 180 cases** so that a second independent healthy opinion can exist.

In [ ]:
AB = pd.DataFrame()

if not judge_A.empty and not judge_B.empty:
    AB = (
        blind_cases
        .merge(
            judge_A[["case_id","verdict","reason","confidence"]],
            on="case_id",
            how="left"
        )
        .rename(columns={
            "verdict":"A_verdict",
            "reason":"A_reason",
            "confidence":"A_confidence"
        })
        .merge(
            judge_B[["case_id","verdict","reason","confidence"]],
            on="case_id",
            how="left"
        )
        .rename(columns={
            "verdict":"B_verdict",
            "reason":"B_reason",
            "confidence":"B_confidence"
        })
    )

healthy_A = HEALTH.get("judge_A",{}).get("eligible",False)
healthy_B = HEALTH.get("judge_B",{}).get("eligible",False)

if healthy_A and healthy_B:
    need_c = (
        ~AB.A_verdict.isin(VALID_VERDICTS) |
        ~AB.B_verdict.isin(VALID_VERDICTS) |
        (AB.A_verdict != AB.B_verdict)
    )
    judge_C_cases = blind_cases[
        blind_cases.case_id.isin(AB.loc[need_c,"case_id"])
    ].copy()
else:
    judge_C_cases = blind_cases.copy()

print("Healthy A:",healthy_A)
print("Healthy B:",healthy_B)
print("Judge C cases:",len(judge_C_cases))

# Part XIV — Run Judge C and assess health

In [ ]:
if RUN_MODELS and len(judge_C_cases):
    # reuse the blind runner but temporarily replace blind_cases inside the call
    original_blind = blind_cases
    blind_cases = judge_C_cases.copy()

    try:
        judge_C = run_blind_judge("judge_C",MODEL_C,303)
    finally:
        blind_cases = original_blind
else:
    judge_C = pd.DataFrame()

if not judge_C.empty:
    HEALTH["judge_C"] = blind_health(
        judge_C,
        CAL_METRICS.get("judge_C",{})
    )

print(json.dumps(HEALTH,indent=2))

health_df = pd.DataFrame([
    {"judge":j,**m}
    for j,m in HEALTH.items()
])

display(health_df)

# Part XV — Consensus using only eligible judges

This is the key v5.1 correction.

A failed or collapsed judge is **not counted as a vote**.

Consensus rules:

- at least two eligible, parsed judge outputs are required;
- two matching eligible votes → consensus;
- three eligible votes with 2-of-3 agreement → consensus;
- otherwise → `UNRESOLVED`.

In [ ]:
def eligible_vote(judge_name, verdict):
    if not HEALTH.get(judge_name,{}).get("eligible",False):
        return None

    return verdict if verdict in VALID_VERDICTS else None

evaluation = blind_cases.copy()

for judge_name,df,prefix in [
    ("judge_A",judge_A,"A"),
    ("judge_B",judge_B,"B"),
    ("judge_C",judge_C,"C")
]:
    if not df.empty:
        small = df[[
            "case_id","verdict","reason","confidence"
        ]].rename(columns={
            "verdict":f"{prefix}_verdict",
            "reason":f"{prefix}_reason",
            "confidence":f"{prefix}_confidence"
        })

        evaluation = evaluation.merge(
            small,
            on="case_id",
            how="left"
        )
    else:
        evaluation[f"{prefix}_verdict"] = np.nan
        evaluation[f"{prefix}_reason"] = np.nan
        evaluation[f"{prefix}_confidence"] = np.nan

def consensus_row(r):
    votes = []

    for j,p in [
        ("judge_A","A"),
        ("judge_B","B"),
        ("judge_C","C")
    ]:
        v = eligible_vote(j,r[f"{p}_verdict"])
        if v is not None:
            votes.append((j,v))

    if len(votes) < 2:
        return pd.Series({
            "consensus_verdict":"UNRESOLVED",
            "consensus_rule":"FEWER_THAN_TWO_ELIGIBLE_JUDGES",
            "eligible_vote_count":len(votes)
        })

    counts = pd.Series([v for _,v in votes]).value_counts()

    if counts.iloc[0] >= 2:
        return pd.Series({
            "consensus_verdict":counts.index[0],
            "consensus_rule":"ELIGIBLE_MAJORITY",
            "eligible_vote_count":len(votes)
        })

    return pd.Series({
        "consensus_verdict":"UNRESOLVED",
        "consensus_rule":"NO_ELIGIBLE_MAJORITY",
        "eligible_vote_count":len(votes)
    })

cons = evaluation.apply(consensus_row,axis=1)
evaluation = pd.concat([evaluation,cons],axis=1)

evaluation = evaluation.merge(
    frozen_reference[[
        "case_id",
        "frozen_prediction",
        "generated_expected"
    ]],
    on="case_id",
    how="left"
)

print(evaluation.consensus_verdict.value_counts(dropna=False))

# Part XVI — Agreement metrics

Metrics are only computed where the compared systems have valid categorical outputs.

In [ ]:
def exact_agreement(a,b):
    a = pd.Series(a)
    b = pd.Series(b)
    ok = a.isin(VALID_VERDICTS) & b.isin(VALID_VERDICTS)

    if not ok.any():
        return np.nan

    return float(
        (a[ok].values == b[ok].values).mean()
    )

def cohens_kappa(a,b):
    labels=["PROVEN","REFUTED","UNPROVEN"]

    a=pd.Series(a)
    b=pd.Series(b)

    ok=a.isin(labels)&b.isin(labels)
    a=a[ok].reset_index(drop=True)
    b=b[ok].reset_index(drop=True)

    if len(a)==0:
        return np.nan

    po=float((a==b).mean())
    pa=a.value_counts(normalize=True)
    pb=b.value_counts(normalize=True)

    pe=sum(
        float(pa.get(x,0))*float(pb.get(x,0))
        for x in labels
    )

    return np.nan if math.isclose(1-pe,0) else (po-pe)/(1-pe)

def fscr(truth,pred):
    truth=pd.Series(truth)
    pred=pd.Series(pred)
    invalid=truth.eq("REFUTED")

    if not invalid.any():
        return np.nan

    return float((pred[invalid]=="PROVEN").mean())

metrics = {
    "consensus_coverage":
        float(evaluation.consensus_verdict.isin(VALID_VERDICTS).mean()),

    "unresolved_cases":
        int((evaluation.consensus_verdict=="UNRESOLVED").sum()),

    "A_vs_generated_accuracy":
        exact_agreement(evaluation.A_verdict,evaluation.generated_expected),

    "B_vs_generated_accuracy":
        exact_agreement(evaluation.B_verdict,evaluation.generated_expected),

    "C_vs_generated_accuracy":
        exact_agreement(evaluation.C_verdict,evaluation.generated_expected),

    "consensus_vs_generated_accuracy":
        exact_agreement(
            evaluation.consensus_verdict,
            evaluation.generated_expected
        ),

    "scigeoguard_vs_consensus_accuracy":
        exact_agreement(
            evaluation.frozen_prediction,
            evaluation.consensus_verdict
        ),

    "kappa_A_B":
        cohens_kappa(evaluation.A_verdict,evaluation.B_verdict),

    "kappa_consensus_scigeoguard":
        cohens_kappa(
            evaluation.consensus_verdict,
            evaluation.frozen_prediction
        ),

    "A_FSCR_vs_generated":
        fscr(evaluation.generated_expected,evaluation.A_verdict),

    "B_FSCR_vs_generated":
        fscr(evaluation.generated_expected,evaluation.B_verdict),

    "C_FSCR_vs_generated":
        fscr(evaluation.generated_expected,evaluation.C_verdict),

    "consensus_FSCR_vs_generated":
        fscr(
            evaluation.generated_expected,
            evaluation.consensus_verdict
        ),

    "scigeoguard_FSCR_vs_generated":
        fscr(
            evaluation.generated_expected,
            evaluation.frozen_prediction
        )
}

metrics_df = pd.DataFrame([
    {"metric":k,"value":v}
    for k,v in metrics.items()
])

display(metrics_df)

# Part XVII — Per-class diagnostics

This explicitly exposes verdict collapse and class-specific failure.

In [ ]:
diagnostics=[]

for name,col in [
    ("Judge_A","A_verdict"),
    ("Judge_B","B_verdict"),
    ("Judge_C","C_verdict"),
    ("Consensus","consensus_verdict"),
    ("SciGeoGuard_X","frozen_prediction")
]:
    for truth in ["PROVEN","REFUTED","UNPROVEN"]:
        g=evaluation[evaluation.generated_expected==truth]

        diagnostics.append({
            "system":name,
            "truth_class":truth,
            "n":len(g),
            "class_recall":(
                float((g[col]==truth).mean())
                if len(g) and col in g else np.nan
            )
        })

diagnostics_df=pd.DataFrame(diagnostics)
display(diagnostics_df)

# Part XVIII — Targeted human audit

The human audit automatically includes:

- all unresolved cases;
- all consensus vs SciGeoGuard-X disagreements;
- all consensus vs generated-reference disagreements;
- random resolved spot checks until at least the target size is reached.

The human-facing CSV contains **no machine verdicts**.

In [ ]:
must_audit = (
    evaluation.consensus_verdict.eq("UNRESOLVED") |
    (
        evaluation.consensus_verdict.isin(VALID_VERDICTS) &
        evaluation.frozen_prediction.ne(
            evaluation.consensus_verdict
        )
    ) |
    (
        evaluation.consensus_verdict.isin(VALID_VERDICTS) &
        evaluation.generated_expected.ne(
            evaluation.consensus_verdict
        )
    )
)

human_internal=evaluation[must_audit].copy()

remaining=evaluation[
    ~evaluation.case_id.isin(human_internal.case_id)
]

needed=max(0,HUMAN_AUDIT_TARGET-len(human_internal))

if needed and len(remaining):
    human_internal=pd.concat([
        human_internal,
        remaining.sample(
            n=min(needed,len(remaining)),
            random_state=SEED
        )
    ],ignore_index=True)

human_blind = blind_cases[
    blind_cases.case_id.isin(human_internal.case_id)
].copy()

for c in [
    "human_verdict",
    "human_error_class",
    "human_severity",
    "human_confidence_1_to_5",
    "human_reason",
    "reviewer_id"
]:
    human_blind[c]=""

print("Human audit size:",len(human_blind))

# Part XIX — Publication figures

In [ ]:
FIGURES=[]

if len(evaluation):
    import matplotlib.pyplot as plt

    systems=["Judge A","Judge B","Judge C","Consensus","SciGeoGuard-X"]

    values=[
        exact_agreement(evaluation.A_verdict,evaluation.generated_expected),
        exact_agreement(evaluation.B_verdict,evaluation.generated_expected),
        exact_agreement(evaluation.C_verdict,evaluation.generated_expected),
        exact_agreement(
            evaluation.consensus_verdict,
            evaluation.generated_expected
        ),
        exact_agreement(
            evaluation.frozen_prediction,
            evaluation.generated_expected
        )
    ]

    p=WORK/"figure_agreement_with_generated_reference.png"
    plt.figure(figsize=(8,4))
    plt.bar(systems,values)
    plt.ylabel("Exact agreement")
    plt.ylim(0,1.05)
    plt.xticks(rotation=20,ha="right")
    plt.tight_layout()
    plt.savefig(p,dpi=220)
    plt.show()
    FIGURES.append(p)

    values=[
        fscr(evaluation.generated_expected,evaluation.A_verdict),
        fscr(evaluation.generated_expected,evaluation.B_verdict),
        fscr(evaluation.generated_expected,evaluation.C_verdict),
        fscr(
            evaluation.generated_expected,
            evaluation.consensus_verdict
        ),
        fscr(
            evaluation.generated_expected,
            evaluation.frozen_prediction
        )
    ]

    p=WORK/"figure_fscr_v5_1.png"
    plt.figure(figsize=(8,4))
    plt.bar(systems,values)
    plt.ylabel("FSCR vs generated reference")
    plt.ylim(0,1.05)
    plt.xticks(rotation=20,ha="right")
    plt.tight_layout()
    plt.savefig(p,dpi=220)
    plt.show()
    FIGURES.append(p)

# Part XX — Protocol-quality gates

In [ ]:
protocol = {
    "blind_input_has_no_generated_truth":
        "generated_expected" not in blind_cases.columns,

    "blind_input_has_no_scigeoguard_output":
        "frozen_prediction" not in blind_cases.columns,

    "blind_input_has_no_family":
        "family" not in blind_cases.columns,

    "calibration_is_separate_from_test":
        set(calibration_df.cal_id).isdisjoint(
            set(blind_cases.case_id)
        ),

    "consensus_requires_two_eligible_judges":
        True,

    "human_file_hides_machine_predictions":
        not any(
            c in human_blind.columns
            for c in [
                "A_verdict",
                "B_verdict",
                "C_verdict",
                "consensus_verdict",
                "frozen_prediction",
                "generated_expected"
            ]
        )
}

protocol_df=pd.DataFrame([
    {"check":k,"passed":bool(v)}
    for k,v in protocol.items()
])

display(protocol_df)

assert protocol_df.passed.all()

# Part XXI — Export results and ZIP

In [ ]:
blind_cases.to_csv(
    WORK/"Blinded_180_cases.csv",
    index=False
)

frozen_reference.to_csv(
    WORK/"Frozen_v3_reference.csv",
    index=False
)

calibration_summary.to_csv(
    WORK/"Calibration_summary.csv",
    index=False
)

health_df.to_csv(
    WORK/"Judge_health_summary.csv",
    index=False
)

evaluation.to_csv(
    WORK/"SciGeoGuard_X_v5_1_evaluation.csv",
    index=False
)

metrics_df.to_csv(
    WORK/"SciGeoGuard_X_v5_1_metrics.csv",
    index=False
)

diagnostics_df.to_csv(
    WORK/"Per_class_diagnostics.csv",
    index=False
)

human_internal.to_csv(
    WORK/"Human_Audit_INTERNAL.csv",
    index=False
)

human_blind.to_csv(
    WORK/"Human_Audit_BLIND.csv",
    index=False
)

protocol_df.to_csv(
    WORK/"Protocol_quality_checks.csv",
    index=False
)

readme = f"""
SciGeoGuard-X Multi-Model Blind Validation v5.1
================================================

Judge A: {MODEL_A}
Judge B: {MODEL_B}
Judge C: {MODEL_C}

Major corrections from v5:
- Phi-3.5 replaced by Phi-4-mini.
- Judge models upgraded to approximately 3B–4B scale.
- Balanced 3-class calibration before blind evaluation.
- Explicit null-field semantics.
- Parse-success and verdict-collapse health gates.
- Failed judges are excluded from consensus.
- Judge C covers all test cases if A or B is unhealthy.
- Consensus requires at least two eligible model votes.
- Human audit remains the final correctness anchor.

Do NOT call multi-model consensus human expert ground truth.

Generated at UTC:
{datetime.now(timezone.utc).isoformat()}
"""

(WORK/"README_v5_1.txt").write_text(
    readme,
    encoding="utf-8"
)

zip_path = shutil.make_archive(
    str(ROOT/"SciGeoGuard_X_v5_1_Results"),
    "zip",
    root_dir=WORK
)

print("Created:",zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Download manually from",zip_path)

# Interpretation

A successful v5.1 run requires more than model agreement.

A judge that:

- cannot parse,
- fails the balanced calibration,
- cannot recognize one of the three classes,
- or collapses almost entirely to one verdict,

is marked unhealthy and is not allowed to determine consensus.

The main quantities to inspect after the run are:

1. calibration accuracy and class recalls;
2. blind parse success;
3. judge-health status;
4. consensus coverage;
5. consensus vs SciGeoGuard-X agreement;
6. consensus vs generated-reference disagreement cases;
7. targeted human-audit cases.

Upload the final v5.1 ZIP for the next analysis.